# W10C1 Lab: Byte Pair Encoding, and What It Costs

Run every cell from the top. **Everything already works.**

Today you will:

1. Watch a real tokenizer break your words apart.
2. Run the merge loop that builds a vocabulary from nothing.
3. See what an instruction-tuned model adds to that vocabulary.
4. Compete: train your team's tokenizer and score it on everyone else's text.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup.
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoTokenizer, logging

logging.set_verbosity_error()
gpt2 = AutoTokenizer.from_pretrained("distilgpt2")
print("GPT-2 vocabulary:", f"{gpt2.vocab_size:,} tokens")

## Part 1. What a tokenizer does to your words

A model has a fixed vocabulary. Anything outside it is chopped into pieces that are inside it.

In [ ]:
# GIVEN. Six words, from ordinary to obscure.
WORDS = ["the", "running", "unhappiness", "tokenization",
         "antidisestablishmentarianism", "Trinity"]

rows = [{"word": w,
         "tokens": len(gpt2.tokenize(w)),
         "split as": " | ".join(gpt2.tokenize(w))} for w in WORDS]
table = pd.DataFrame(rows)
print(table.to_string(index=False))

plt.figure(figsize=(7, 2.8))
plt.bar(table["word"], table["tokens"], color="#7C2529")
plt.ylabel("tokens"); plt.xticks(rotation=30, ha="right")
plt.title("One word is not one token"); plt.tight_layout(); plt.show()

In [ ]:
# ================== TRY IT 1 ==================
# Put your own words in WORDS: your name, a word in another language, an emoji, a long ordinary English word.
# Which of them are expensive, and what do the cheap ones have in common?
# ==============================================


## Part 2. Building a vocabulary from nothing

BPE starts from single characters and repeatedly glues together the commonest adjacent pair. That is the whole algorithm.

In [ ]:
# GIVEN. The merge loop, printed as it runs.
CORPUS = {"low": 5, "lower": 2, "newest": 6, "widest": 3}

def commonest_pair(words):
    pairs = Counter()
    for word, freq in words.items():
        symbols = word.split()
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs.most_common(1)[0] if pairs else (None, 0)

def merge_loop(n_merges, show=True):
    words = {" ".join(w) + " </w>": n for w, n in CORPUS.items()}
    learned = []
    for step in range(n_merges):
        pair, count = commonest_pair(words)
        if pair is None:
            break
        words = {w.replace(" ".join(pair), "".join(pair)): n for w, n in words.items()}
        learned.append("".join(pair))
        if show:
            print(f"merge {step + 1:>2}: {pair[0]!r} + {pair[1]!r} "
                  f"-> {''.join(pair)!r}  (seen {count} times)")
    lengths = [len(w.split()) for w in words]
    print(f"\nnew tokens learned: {len(learned)}")
    print(f"average tokens per word: {sum(lengths) / len(lengths):.2f}")
    for w in words:
        print("   ", w)
    return learned

merge_loop(5)

In [ ]:
# ================== TRY IT 2 ==================
# Call merge_loop with 2 merges, then with 30.
# Which way does the average tokens per word move, and what do you pay for it?
# ==============================================


## Part 3. What instruction tuning adds

Instruction tuning is the same next-token loss on different data: (instruction, response) pairs, written into a template.

In [ ]:
# GIVEN. An instruction-tuned model's tokenizer, and the template it was trained on.
qwen = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

pair = [{"role": "user", "content": "Explain the moon landing to a six-year-old."},
        {"role": "assistant", "content": "People flew there, took pictures, "
                                         "and sent them back."}]

text = qwen.apply_chat_template(pair, tokenize=False)
print(text)
print("that is", len(qwen.encode(text)), "tokens of ordinary training text\n")

# The template's markers are not text to this model. They are vocabulary.
for marker in ["<|im_start|>", "<|im_end|>"]:
    print(f"{marker:<14} Qwen: {len(qwen.tokenize(marker))} token"
          f"   GPT-2: {len(gpt2.tokenize(marker))} tokens")

In [ ]:
# ================== TRY IT 3 ==================
# Replace the pair with an instruction and response of your own, and print the template again.
# The model is trained to predict every token in this string. Which of them do you actually want it to learn?
# ==============================================


## Part 4. The tokenizer bake-off (teams)

One team, one corpus. Train your BPE on it, then measure how many tokens it needs for every other team's text. Lowest is best.

In [ ]:
# GIVEN. A real BPE trainer and encoder, and five domain corpora.
from data.corpora import CORPORA

END = "</w>"
DOMAINS = list(CORPORA)

def train_bpe(text, n_merges):
    vocab = Counter(tuple(w) + (END,) for w in text.lower().split())
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for symbols, freq in vocab.items():
            for ab in zip(symbols, symbols[1:]):
                pairs[ab] += freq
        if not pairs:
            break
        best = max(pairs, key=lambda ab: (pairs[ab], ab))   # ties: larger pair wins
        merges.append(best)
        vocab = Counter({apply_merge(best, s): f for s, f in vocab.items()})
    return merges

def apply_merge(pair, symbols):
    a, b = pair
    out, i = [], 0
    while i < len(symbols):
        if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
            out.append(a + b); i += 2
        else:
            out.append(symbols[i]); i += 1
    return tuple(out)

def encode(word, merges):
    symbols = tuple(word.lower()) + (END,)
    for pair in merges:
        symbols = apply_merge(pair, symbols)
    return symbols

def tokens_per_word(text, merges):
    words = text.lower().split()
    return sum(len(encode(w, merges)) for w in words) / len(words)

print("corpora:", ", ".join(DOMAINS))

In [ ]:
# ================== YOUR TURN 1 ==================
# Claim your team's corpus, train on it, and encode a word from it.
#
# Expected: english + 300 merges -> 256 merges learned (it runs out of pairs),
#           and 'river' encodes as a single piece
# =================================================
MY_DOMAIN = "english"        # <-- english, python, spanish, chat, chemistry
N_MERGES = 300

merges = train_bpe(CORPORA[MY_DOMAIN]["train"], N_MERGES)
print(f"{MY_DOMAIN}: asked for {N_MERGES} merges, learned {len(merges)}")
print("last five merges:", ["".join(m) for m in merges[-5:]])
print("'river' ->", encode("river", merges))

In [ ]:
# ================== YOUR TURN 2 ==================
# Score your tokenizer on every team's held-out text, then post your
# own-domain number and your average on the board.
#
# Expected: the english tokenizer scores about 1.33 on english, 3.65 on spanish,
#           5.39 on chemistry and 6.17 on python, averaging 3.83
# =================================================
scores = {d: tokens_per_word(CORPORA[d]["test"], merges) for d in DOMAINS}

for d, s in scores.items():
    mark = "  <- your own" if d == MY_DOMAIN else ""
    print(f"   {d:<10} {s:5.2f} tokens per word{mark}")
print(f"\n   average  {sum(scores.values()) / len(scores):5.2f}")

colors = ["#7C2529" if d == MY_DOMAIN else "#d2d2d7" for d in DOMAINS]
plt.figure(figsize=(7, 2.8))
plt.bar(list(scores), list(scores.values()), color=colors)
plt.ylabel("tokens per word"); plt.title(f"'{MY_DOMAIN}' tokenizer on everyone's text")
plt.tight_layout(); plt.show()

In [ ]:
# ================== YOUR TURN 3 ==================
# Buy a bigger vocabulary and see who it helps.
#
# Expected: english's own score falls from 2.29 to 1.33 between 100 and 900
#           merges, while its python score barely moves, 6.56 to 6.17
# =================================================
BUDGETS = [100, 300, 900]    # <-- a vocabulary is bought in merges

for n in BUDGETS:
    m = train_bpe(CORPORA[MY_DOMAIN]["train"], n)
    own = tokens_per_word(CORPORA[MY_DOMAIN]["test"], m)
    other = "python" if MY_DOMAIN != "python" else "english"
    print(f"   {n:>3} merges ({len(m):>3} learned):  own {own:.2f}   {other} "
          f"{tokens_per_word(CORPORA[other]['test'], m):.2f}")

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   Ordinary English words are one token however long they are; "running" and
#   "photosynthesis" both cost 1. Names, other languages and emoji cost 3 to 10.
#   The bill is not proportional to how much you wrote, it is proportional to how
#   ORDINARY what you wrote was.
#
# TRY IT 2
#   2 merges: almost everything is still characters, so sequences are long.
#   30 merges: whole words become single tokens and sequences get short. What you
#   pay is vocabulary size, and every entry is a row in the model's embedding
#   matrix and its output layer. That trade is the whole design decision.
#
# TRY IT 3
#   You want it to learn the RESPONSE. The instruction is the prompt, and it is
#   given at inference time, so training loss is masked to the response tokens.
#   The markers are one token in Qwen and seven in GPT-2, because an
#   instruction-tuned model adds them to its vocabulary on purpose: they are the
#   only thing separating "who is speaking" from ordinary text.
#
# YOUR TURN 1
#   MY_DOMAIN = "chat" learns only 193 merges out of 300: the corpus runs out of
#   adjacent pairs to merge. A vocabulary cannot be larger than its data supports.
#
# YOUR TURN 2
#   Every tokenizer wins its own column and loses everywhere else. The english
#   tokenizer needs 2.7x as many tokens for spanish and 4.6x for python. That gap
#   is what a non-English user pays: same meaning, more tokens, smaller effective
#   context, larger bill.
#
# YOUR TURN 3
#   More merges help the domain you trained on and do almost nothing for the
#   others. A shipped model gets ONE vocabulary, so its tokenizer encodes a
#   decision about whose text is cheap.